# Step 0 — Prepare your Jupyter environment

**Goal:** Select one isolated Python kernel and check Python libraries, OGS command-line programs and existing model files **before** the training begins.

**Recommended:** Run `bash setup_jupyter.sh` in the VS Code terminal **before the meeting** with working internet access. Then choose **OGS Workshop (Python 3.12)** in the notebook kernel selector.

> The colored kernel selection only identifies the interpreter. It does not prove packages and OGS binaries are available.


In [ ]:
import sys, pathlib, importlib, shutil
from IPython.display import display, Markdown
print('Active Python:',sys.executable)
print('Python version:',sys.version.split()[0])
checks={}
for module in ('numpy','matplotlib','meshio','scipy','vtk','pyvista','vtuIO','ogs6py.ogs','gmsh','ipykernel'):
    try:
        importlib.import_module(module)
        checks[module]=True
    except Exception as exc:
        checks[module]=False
        print(f'FAIL {module}: {exc}')
for name in ('ogs','GMSH2OGS'):
    checks[name]=shutil.which(name) is not None
print('\n'.join(f"{'PASS' if good else 'FAIL'}: {key}" for key,good in checks.items()))
assert all(checks.values()),'Some requirements are missing. Run setup_jupyter.sh or ask the instructor.'
print('PASS: Python libraries and OGS tools detected')


## Plot real OGS output

**Explain:** A VTU file is a native OpenGeoSys finite-element result. This plot shows a real potential field, not a generic sample image. **No OGS computation is launched in Step 0.**


In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib.tri as mtri, meshio
from pathlib import Path
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'models'/'excavation.prj').exists()),None)
assert ROOT is not None,'Open the workshop folder in VS Code'
files=sorted((ROOT/'results').glob('excavation_diffusion_ts_1_*.vtu'))
assert files,'Precomputed reference VTU missing'
m=meshio.read(files[-1]); triangles=np.vstack([c.data for c in m.cells if c.type=='triangle'])
u=np.asarray(m.point_data['pressure']).reshape(-1)
fig,ax=plt.subplots(figsize=(10,4.6))
cf=ax.tricontourf(m.points[:,0],m.points[:,1],triangles,u,levels=np.linspace(30,60,25),cmap='coolwarm')
fig.colorbar(cf,ax=ax,label='Potential [model units]')
ax.set(xlabel='x [m]',ylabel='y [m]',title='OGS reference simulation — VTU read & plot check')
ax.set_aspect('equal');plt.show()
assert np.isfinite(u).all()
print(f'PASS: loaded {files[-1].name}; {len(m.points)} nodes; range {u.min():.4g} to {u.max():.4g}')


## Instructor checklist

A green final PASS and a visible potential contour plot mean the **Python/Jupyter post-processing path** is operational. The external `ogs` and `GMSH2OGS` programs are only checked for presence; **Step 0 is not a full native simulation test**. Run the entire workflow on the teaching machine before the meeting.
